# FEDE Embedding Fine-Tuning Pipeline

Run each stage in order. Each cell is independent — you can re-run any stage without repeating earlier ones.

| Stage | What it does |
|---|---|
| 0 | Sanity checks |
| 1a | Generate queries (LLM only, no embedding model) |
| 1b | Assemble training pairs (embedding + positive assignment) |
| 2 | Generate eval queries |
| 3 | Train Round 1 |
| 4 | Mine hard negatives |
| 5 | Train Round 2 |
| 6 | Evaluate |
| 7 | Index corpus into Qdrant |

## Configuration

Edit these before running anything.

In [1]:
# ── How many movies to use for training (rest become eval pool) ──────────────
TRAIN_MOVIES     = 1200
N_EVAL_QUERIES   = 100       # held-out eval queries to generate

# ── Output directories ───────────────────────────────────────────────────────
ROUND1_MODEL_DIR = "fede-embeddinggemma/round1"
ROUND2_MODEL_DIR = "fede-embeddinggemma/round2"

# ── Training overrides (None = use config.py defaults) ───────────────────────
EPOCHS      = None   # e.g. 1 for a quick test
BATCH_SIZE  = 8      # per-device batch; try 2 on 8–12GB if you OOM
LR          = None   # e.g. 1e-5
USE_LORA    = True   # False = full fine-tune (needs more VRAM)
CACHED_MNRL_MINI_BATCH = 8   # CachedMNRL sub-batch; lower if OOM (e.g. 4)

# ── Evaluation ───────────────────────────────────────────────────────────────
EVAL_MOVIES = None   # None = full 1338-movie corpus; set e.g. 500 to save RAM

---
## Stage 0 — Sanity checks

Verify environment, credentials, and data before committing hours of compute.

In [2]:
import logging, os, json
from pathlib import Path

logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s")
log = logging.getLogger("pipeline")

# ── Project root ─────────────────────────────────────────────────────────────
import finetuning.config as cfg
log.info("Project root: %s", cfg.PROJECT_ROOT)

# ── Tagged scripts ────────────────────────────────────────────────────────────
tagged_files = list(cfg.TAGGED_SCRIPTS_DIR.glob("*.txt"))
log.info("Tagged scripts found: %d", len(tagged_files))
assert len(tagged_files) > 0, f"No tagged scripts in {cfg.TAGGED_SCRIPTS_DIR}"

# ── Metadata ─────────────────────────────────────────────────────────────────
with open(cfg.METADATA_PATH) as f:
    meta = json.load(f)
log.info("Metadata entries: %d", len(meta))

# ── Gemini API key (used by Batch API for dataset generation) ─────────────────
gemini_key = os.getenv("GEMINI_API_KEY", "")
assert gemini_key, "GEMINI_API_KEY is not set in .env"
log.info("Gemini key: %s…%s", gemini_key[:8], gemini_key[-4:])

# ── HF token (for gated embedding model) ─────────────────────────────────────
hf_token = os.getenv("HF_TOKEN", "")
if hf_token:
    log.info("HF_TOKEN present")
else:
    log.warning("HF_TOKEN not set — fine unless you switched to a gated model")

# ── Existing data ─────────────────────────────────────────────────────────────
raw_path  = cfg.FINETUNING_DATA_DIR / "raw_queries.jsonl"
r1_path   = cfg.FINETUNING_DATA_DIR / "training_pairs_r1.jsonl"
r2_path   = cfg.FINETUNING_DATA_DIR / "training_pairs_r2.jsonl"
eval_path = cfg.FINETUNING_DATA_DIR / "eval_queries.json"
ckpt_path = cfg.FINETUNING_DATA_DIR / "querygen_checkpoint.json"

def _count(p): return sum(1 for _ in open(p)) if p.exists() else 0

print(f"\nCurrent data state:")
print(f"  raw_queries.jsonl       : {_count(raw_path):>6} lines")
print(f"  training_pairs_r1.jsonl : {_count(r1_path):>6} lines")
print(f"  training_pairs_r2.jsonl : {_count(r2_path):>6} lines")
print(f"  eval_queries.json       : {'exists' if eval_path.exists() else 'missing'}")
print(f"  querygen_checkpoint.json: {'exists' if ckpt_path.exists() else 'missing'}")
if ckpt_path.exists():
    ckpt = json.loads(ckpt_path.read_text())
    print(f"    → {len(ckpt.get('processed_movies', []))} movies in checkpoint, status={ckpt.get('status', 'in_progress')}")

print("\n✅ Sanity checks passed")

2026-04-09 01:27:45,897 [INFO] Project root: C:\Users\anzez\University\ML\fede
2026-04-09 01:27:45,932 [INFO] Tagged scripts found: 1338
2026-04-09 01:27:45,949 [INFO] Metadata entries: 1384
2026-04-09 01:27:45,950 [INFO] Gemini key: AIzaSyC1…QQDE
2026-04-09 01:27:45,950 [INFO] HF_TOKEN present



Current data state:
  raw_queries.jsonl       :   6946 lines
  training_pairs_r1.jsonl :   7651 lines
  training_pairs_r2.jsonl :   7651 lines
  eval_queries.json       : exists
  querygen_checkpoint.json: exists
    → 1200 movies in checkpoint, status=queries_complete

✅ Sanity checks passed


---
## Stage 1a — Generate queries (LLM only)

Calls the LLM for every movie to produce Type A (synopsis) and Type B (scene summary) queries.
No embedding model is loaded during this stage — only API calls.

- Checkpoints every 50 movies. Safe to interrupt and resume.
- Output: `data/finetuning/raw_queries.jsonl`
- Uses structured output (Pydantic schemas) to guarantee valid JSON from the LLM.

In [ ]:
# ── Stage 1a: Generate queries via LLM (no embedding model) ─────────────────
from finetuning.dataset.dataset_builder import DatasetBuilder

builder = DatasetBuilder(max_movies=TRAIN_MOVIES)
raw_queries_path = builder.generate_queries(resume=True)

n_queries = sum(1 for _ in open(raw_queries_path))
print(f"\n✅ Stage 1a complete: {n_queries} raw queries → {raw_queries_path}")
print("You can now safely restart the kernel before running Stage 1b.")

In [3]:
import gc

# Free QueryGenerator / LLM client before loading the embedding model.
if "builder" in dir():
    del builder
gc.collect()
print("Stage 1a objects released")

Stage 1a objects released


In [4]:
# ── Stage 1b: Assemble training pairs (embedding + positive assignment) ──────
# Reads raw_queries.jsonl, loads embedding model, finds best positive scene
# per query, samples random negatives, and writes training_pairs_r1.jsonl.
#
# No LLM API calls — purely local compute.
# Safe to interrupt and resume — already-processed movies are skipped.

from finetuning.dataset.dataset_builder import DatasetBuilder

builder = DatasetBuilder(max_movies=TRAIN_MOVIES)
r1_path = builder.assemble_pairs(resume=True)

pairs = sum(1 for _ in open(r1_path))
print(f"\n✅ Stage 1b complete: {pairs} training pairs → {r1_path}")

2026-04-08 17:03:04,370 [INFO] Scene corpus built: 1200 movies, 117100 total scenes
2026-04-08 17:03:04,371 [INFO] Loading embedding model google/embeddinggemma-300m on cuda …
2026-04-08 17:03:04,371 [INFO] Loading embedding model: google/embeddinggemma-300m
2026-04-08 17:03:04,374 [INFO] Load pretrained SentenceTransformer: google/embeddinggemma-300m
2026-04-08 17:03:04,772 [INFO] HTTP Request: HEAD https://huggingface.co/google/embeddinggemma-300m/resolve/main/modules.json "HTTP/1.1 200 OK"
2026-04-08 17:03:04,897 [INFO] HTTP Request: HEAD https://huggingface.co/google/embeddinggemma-300m/resolve/main/config_sentence_transformers.json "HTTP/1.1 200 OK"
2026-04-08 17:03:05,024 [INFO] HTTP Request: HEAD https://huggingface.co/google/embeddinggemma-300m/resolve/main/config_sentence_transformers.json "HTTP/1.1 200 OK"
2026-04-08 17:03:05,141 [INFO] HTTP Request: HEAD https://huggingface.co/google/embeddinggemma-300m/resolve/main/README.md "HTTP/1.1 200 OK"
2026-04-08 17:03:05,272 [INFO] 

FileExistsError: [WinError 183] Cannot create a file when that file already exists: 'C:\\Users\\anzez\\University\\ML\\fede\\data\\finetuning\\querygen_checkpoint.tmp' -> 'C:\\Users\\anzez\\University\\ML\\fede\\data\\finetuning\\querygen_checkpoint.json'

In [5]:
import gc

# Free embedding model + corpus from Stage 1b before loading training model.
if "builder" in dir():
    del builder
gc.collect()
print("Stage 1b objects released")

Stage 1b objects released


In [ ]:
# (Optional) Clear the checkpoint to force a full re-run of Stage 1a
# import os
# ckpt = cfg.FINETUNING_DATA_DIR / "querygen_checkpoint.json"
# if ckpt.exists():
#     os.remove(ckpt)
#     print("Checkpoint cleared")

In [ ]:
# (Legacy) Run both stages in one call — use Stage 1a + 1b cells above instead.
# from finetuning.dataset.dataset_builder import DatasetBuilder
# builder = DatasetBuilder(max_movies=TRAIN_MOVIES)
# r1_path = builder.build(resume=True)
# pairs = sum(1 for _ in open(r1_path))
# print(f"\n✅ Stage 1 complete: {pairs} pairs → {r1_path}")

---
## Stage 2 — Generate eval queries

Generates ~100 held-out evaluation queries from movies **not** in the training set.

⚠️ Must run **after Stage 1** — reads the checkpoint to know which movies to exclude.

In [7]:
from finetuning.corpus.scene_corpus import load_metadata
from finetuning.dataset.query_generator import load_checkpoint
from finetuning.evaluation.dataset_generator import generate_eval_dataset

ckpt = load_checkpoint()
assert ckpt, "No checkpoint found — run Stage 1 first"
training_ids = set(ckpt.get("processed_movies", []))
log.info("Excluding %d training movies from eval pool", len(training_ids))

metadata = load_metadata()
eval_path = generate_eval_dataset(
    corpus_movie_ids=training_ids,
    metadata=metadata,
    n=N_EVAL_QUERIES,
)

with open(eval_path) as f:
    eval_queries = json.load(f)
print(f"\n✅ Stage 2 complete: {len(eval_queries)} eval queries → {eval_path}")
print("Sample:", eval_queries[0])

2026-03-22 21:25:19,391 [INFO] Excluding 1200 training movies from eval pool
2026-03-22 21:25:19,666 [INFO] Generating eval queries from 173 candidate movies (target: 100)
2026-03-22 21:25:24,962 [INFO] HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/openai/chat/completions "HTTP/1.1 200 OK"
2026-03-22 21:25:27,826 [INFO] HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/openai/chat/completions "HTTP/1.1 200 OK"
2026-03-22 21:25:31,396 [INFO] HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/openai/chat/completions "HTTP/1.1 200 OK"
2026-03-22 21:25:36,323 [INFO] HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/openai/chat/completions "HTTP/1.1 200 OK"
2026-03-22 21:25:38,631 [INFO] HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/openai/chat/completions "HTTP/1.1 200 OK"
2026-03-22 21:25:42,140 [INFO] HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/openai/chat/completions 


✅ Stage 2 complete: 100 eval queries → C:\Users\anzez\University\ML\fede\data\finetuning\eval_queries.json
Sample: {'query': 'A district attorney leads an extensive investigation into the assassination of a national president.', 'movie_id': 'jfk', 'movie_title': 'JFK'}


In [ ]:
# (Cleanup already handled by cells above)

---
## Stage 3 — Train Round 1

Fine-tunes the base embedding model on the round-1 dataset using `CachedMultipleNegativesRankingLoss`.
Runs retrieval eval (MRR / Accuracy@k) after each epoch if `eval_queries.json` exists.

💡 **VRAM:** Use small `BATCH_SIZE` (2–4), `USE_LORA = True`, and low `CACHED_MNRL_MINI_BATCH` (4–8) on 8–12GB GPUs. Optionally set `PYTORCH_ALLOC_CONF=expandable_segments:True` before starting Jupyter.

In [6]:
from finetuning.training.model import load_model
from finetuning.training.trainer import build_evaluator, build_trainer, load_training_dataset
from finetuning.corpus.scene_corpus import build_scene_corpus

model      = load_model()  # base model from config
train_data = load_training_dataset(r1_path)
log.info("Training samples: %d", len(train_data))

evaluator = None
if eval_path.exists():
    log.info("Building eval corpus for per-epoch evaluation …")
    eval_corpus = build_scene_corpus(max_movies=EVAL_MOVIES)
    evaluator = build_evaluator(eval_path, eval_corpus)
else:
    log.warning("eval_queries.json not found — training without per-epoch eval")

train_kwargs = {}
if EPOCHS     is not None: train_kwargs["num_epochs"]    = EPOCHS
if BATCH_SIZE is not None: train_kwargs["batch_size"]    = BATCH_SIZE
if LR         is not None: train_kwargs["learning_rate"] = LR

trainer = build_trainer(
    model=model,
    train_dataset=train_data,
    output_dir=ROUND1_MODEL_DIR,
    evaluator=evaluator,
    use_lora=USE_LORA,
    cached_mnrl_mini_batch=CACHED_MNRL_MINI_BATCH,
    **train_kwargs,
)

trainer.train()
model.save(ROUND1_MODEL_DIR)
print(f"\n✅ Stage 3 complete: round-1 model saved to {ROUND1_MODEL_DIR}")

2026-04-08 17:15:52,971 [INFO] Loading embedding model: google/embeddinggemma-300m
2026-04-08 17:15:52,999 [INFO] Use pytorch device_name: cuda:0
2026-04-08 17:15:53,000 [INFO] Load pretrained SentenceTransformer: google/embeddinggemma-300m
2026-04-08 17:15:53,135 [INFO] HTTP Request: HEAD https://huggingface.co/google/embeddinggemma-300m/resolve/main/modules.json "HTTP/1.1 200 OK"
2026-04-08 17:15:53,267 [INFO] HTTP Request: HEAD https://huggingface.co/google/embeddinggemma-300m/resolve/main/config_sentence_transformers.json "HTTP/1.1 200 OK"
2026-04-08 17:15:53,394 [INFO] HTTP Request: HEAD https://huggingface.co/google/embeddinggemma-300m/resolve/main/config_sentence_transformers.json "HTTP/1.1 200 OK"
2026-04-08 17:15:53,510 [INFO] HTTP Request: HEAD https://huggingface.co/google/embeddinggemma-300m/resolve/main/README.md "HTTP/1.1 200 OK"
2026-04-08 17:15:53,644 [INFO] HTTP Request: HEAD https://huggingface.co/google/embeddinggemma-300m/resolve/main/modules.json "HTTP/1.1 200 OK"


Epoch,Training Loss,Validation Loss,Fede-ir-eval Cosine Accuracy@5,Fede-ir-eval Cosine Accuracy@10,Fede-ir-eval Cosine Accuracy@20,Fede-ir-eval Cosine Precision@5,Fede-ir-eval Cosine Precision@10,Fede-ir-eval Cosine Precision@20,Fede-ir-eval Cosine Recall@5,Fede-ir-eval Cosine Recall@10,Fede-ir-eval Cosine Recall@20,Fede-ir-eval Cosine Ndcg@20,Fede-ir-eval Cosine Mrr@20,Fede-ir-eval Cosine Map@20
1,0.463718,No log,0.679348,0.744565,0.793478,0.295652,0.234783,0.178804,0.020677,0.033384,0.049370,0.227588,0.559183,0.126534
2,0.335130,No log,0.684783,0.744565,0.804348,0.302174,0.234783,0.181793,0.022147,0.033387,0.049849,0.231669,0.578358,0.128313


2026-04-08 17:38:38,836 [INFO] Information Retrieval Evaluation of the model on the fede-ir-eval dataset in epoch 1.0 after 957 steps:
2026-04-08 17:48:04,979 [INFO] Queries: 184
2026-04-08 17:48:04,979 [INFO] Corpus: 123397

2026-04-08 17:48:04,991 [INFO] Score-Function: cosine
2026-04-08 17:48:04,991 [INFO] Accuracy@5: 67.93%
2026-04-08 17:48:04,992 [INFO] Accuracy@10: 74.46%
2026-04-08 17:48:04,992 [INFO] Accuracy@20: 79.35%
2026-04-08 17:48:04,992 [INFO] Precision@5: 29.57%
2026-04-08 17:48:04,992 [INFO] Precision@10: 23.48%
2026-04-08 17:48:04,993 [INFO] Precision@20: 17.88%
2026-04-08 17:48:04,993 [INFO] Recall@5: 2.07%
2026-04-08 17:48:04,993 [INFO] Recall@10: 3.34%
2026-04-08 17:48:04,994 [INFO] Recall@20: 4.94%
2026-04-08 17:48:04,994 [INFO] MRR@20: 0.5592
2026-04-08 17:48:04,994 [INFO] NDCG@20: 0.2276
2026-04-08 17:48:04,995 [INFO] MAP@20: 0.1265
2026-04-08 17:48:04,998 [INFO] Saving model checkpoint to fede-embeddinggemma/round1\checkpoint-957
2026-04-08 17:48:04,999 [INFO] 


✅ Stage 3 complete: round-1 model saved to fede-embeddinggemma/round1


In [7]:
import gc

# Free training objects before loading the corpus + model for hard-negative mining.
for _var in ("model", "trainer", "train_data", "eval_corpus", "evaluator"):
    if _var in globals():
        del globals()[_var]
gc.collect()
print("🧹 Stage 3 objects released")

🧹 Stage 3 objects released


---
## Stage 4 — Mine hard negatives

Uses the round-1 model to replace random negatives with semantically close (but wrong) scenes.
This makes round-2 training significantly harder and usually gives the biggest quality jump.

⚠️ Peak RAM: loads round-1 model + full scene corpus simultaneously.
Pass `--movies N` equivalent by setting `EVAL_MOVIES` above if you OOM.

In [8]:
from finetuning.dataset.negative_miner import CorpusIndex, mine_hard_negatives

r1_model = load_model(ROUND1_MODEL_DIR)

log.info("Building scene corpus …")
corpus = build_scene_corpus(max_movies=EVAL_MOVIES)

log.info("Building embedding index …")
index = CorpusIndex.build(corpus, r1_model)

log.info("Loading round-1 dataset …")
with open(r1_path) as f:
    r1_rows = [json.loads(line) for line in f]

log.info("Mining hard negatives for %d pairs …", len(r1_rows))
r2_path.parent.mkdir(parents=True, exist_ok=True)
with open(r2_path, "w", encoding="utf-8") as out:
    for i, row in enumerate(r1_rows, 1):
        hard_negs = mine_hard_negatives(
            query=row["anchor"],
            movie_id=row["movie_id"],
            index=index,
            model=r1_model,
            n=cfg.HARD_NEGATIVES_PER_QUERY,
        )
        row["negatives"] = hard_negs
        out.write(json.dumps(row, ensure_ascii=False) + "\n")
        if i % 500 == 0:
            log.info("  %d / %d", i, len(r1_rows))

print(f"\n✅ Stage 4 complete: {len(r1_rows)} pairs with hard negatives → {r2_path}")

2026-04-08 18:20:43,062 [INFO] Loading embedding model: fede-embeddinggemma/round1
Loading weights: 100%|██████████| 314/314 [00:00<00:00, 2608.22it/s]
2026-04-08 18:20:47,979 [INFO] Use pytorch device_name: cuda:0
2026-04-08 18:20:48,063 [INFO] Model loaded — device=cuda:0, dim=768
2026-04-08 18:20:48,064 [INFO] Building scene corpus …
2026-04-08 18:20:52,380 [INFO] Scene corpus built: 1262 movies, 123397 total scenes
2026-04-08 18:20:52,382 [INFO] Building embedding index …
2026-04-08 18:20:52,401 [INFO] Encoding 123397 scenes for hard-negative index …
Batches: 100%|██████████| 965/965 [10:00<00:00,  1.61it/s]
2026-04-08 18:30:57,779 [INFO] Loading round-1 dataset …
2026-04-08 18:30:58,011 [INFO] Mining hard negatives for 7651 pairs …
2026-04-08 18:31:45,747 [INFO]   500 / 7651
2026-04-08 18:32:32,881 [INFO]   1000 / 7651
2026-04-08 18:33:19,150 [INFO]   1500 / 7651
2026-04-08 18:34:05,493 [INFO]   2000 / 7651
2026-04-08 18:34:52,001 [INFO]   2500 / 7651
2026-04-08 18:35:37,022 [INFO


✅ Stage 4 complete: 7651 pairs with hard negatives → C:\Users\anzez\University\ML\fede\data\finetuning\training_pairs_r2.jsonl


---
## Stage 5 — Train Round 2

Continues fine-tuning from the round-1 checkpoint using the hard-negative dataset.

In [9]:
r2_model   = load_model(ROUND1_MODEL_DIR)  # start from round-1 weights
train_data2 = load_training_dataset(r2_path)
log.info("Round-2 training samples: %d", len(train_data2))

evaluator2 = None
if eval_path.exists():
    eval_corpus2 = build_scene_corpus(max_movies=EVAL_MOVIES)
    evaluator2 = build_evaluator(eval_path, eval_corpus2)

r2_train_kwargs = dict(train_kwargs)
r2_train_kwargs.setdefault("learning_rate", cfg.ROUND2_LEARNING_RATE)

trainer2 = build_trainer(
    model=r2_model,
    train_dataset=train_data2,
    output_dir=ROUND2_MODEL_DIR,
    evaluator=evaluator2,
    use_lora=USE_LORA,
    cached_mnrl_mini_batch=CACHED_MNRL_MINI_BATCH,
    **r2_train_kwargs,
)

trainer2.train()
r2_model.save(ROUND2_MODEL_DIR)
print(f"\n✅ Stage 5 complete: round-2 model saved to {ROUND2_MODEL_DIR}")

2026-04-08 18:43:16,811 [INFO] Loading embedding model: fede-embeddinggemma/round1
Loading weights: 100%|██████████| 314/314 [00:00<00:00, 3283.04it/s]
2026-04-08 18:43:20,226 [INFO] Use pytorch device_name: cuda:0
2026-04-08 18:43:20,308 [INFO] Model loaded — device=cuda:0, dim=768
2026-04-08 18:43:20,646 [INFO] Loaded 7651 training pairs from C:\Users\anzez\University\ML\fede\data\finetuning\training_pairs_r2.jsonl
2026-04-08 18:43:21,064 [INFO] Round-2 training samples: 7651
2026-04-08 18:43:25,411 [INFO] Scene corpus built: 1262 movies, 123397 total scenes
2026-04-08 18:43:25,476 [INFO] Evaluator built — 184 queries, 123397 scene docs (from 1262 movies), k=[5, 10, 20]
2026-04-08 18:43:25,482 [INFO] Model already has LoRA adapters loaded; reusing existing PEFT model
2026-04-08 18:43:25,483 [WARNING] The `warmup_ratio` argument is deprecated in Transformers v5+, and will also be removed from Sentence Transformers once support for Transformers v4 is dropped. Since you're using Transfo

Epoch,Training Loss,Validation Loss,Fede-ir-eval Cosine Accuracy@5,Fede-ir-eval Cosine Accuracy@10,Fede-ir-eval Cosine Accuracy@20,Fede-ir-eval Cosine Precision@5,Fede-ir-eval Cosine Precision@10,Fede-ir-eval Cosine Precision@20,Fede-ir-eval Cosine Recall@5,Fede-ir-eval Cosine Recall@10,Fede-ir-eval Cosine Recall@20,Fede-ir-eval Cosine Ndcg@20,Fede-ir-eval Cosine Mrr@20,Fede-ir-eval Cosine Map@20
1,2.400150,No log,0.711957,0.771739,0.809783,0.296739,0.232609,0.177717,0.019707,0.032993,0.050028,0.228962,0.594422,0.123093
2,2.215259,No log,0.706522,0.766304,0.815217,0.294565,0.229891,0.177989,0.019659,0.032767,0.050313,0.228269,0.588597,0.122179


2026-04-08 19:05:33,777 [INFO] Information Retrieval Evaluation of the model on the fede-ir-eval dataset in epoch 1.0 after 957 steps:
2026-04-08 19:14:50,857 [INFO] Queries: 184
2026-04-08 19:14:50,858 [INFO] Corpus: 123397

2026-04-08 19:14:50,870 [INFO] Score-Function: cosine
2026-04-08 19:14:50,870 [INFO] Accuracy@5: 71.20%
2026-04-08 19:14:50,870 [INFO] Accuracy@10: 77.17%
2026-04-08 19:14:50,870 [INFO] Accuracy@20: 80.98%
2026-04-08 19:14:50,871 [INFO] Precision@5: 29.67%
2026-04-08 19:14:50,871 [INFO] Precision@10: 23.26%
2026-04-08 19:14:50,871 [INFO] Precision@20: 17.77%
2026-04-08 19:14:50,872 [INFO] Recall@5: 1.97%
2026-04-08 19:14:50,872 [INFO] Recall@10: 3.30%
2026-04-08 19:14:50,872 [INFO] Recall@20: 5.00%
2026-04-08 19:14:50,873 [INFO] MRR@20: 0.5944
2026-04-08 19:14:50,873 [INFO] NDCG@20: 0.2290
2026-04-08 19:14:50,873 [INFO] MAP@20: 0.1231
2026-04-08 19:14:50,877 [INFO] Saving model checkpoint to fede-embeddinggemma/round2\checkpoint-957
2026-04-08 19:14:50,878 [INFO] 


✅ Stage 5 complete: round-2 model saved to fede-embeddinggemma/round2


In [10]:
import gc

# Free the corpus index + r1_model before loading the training model for Round 2.
# The CorpusIndex holds all 117k scene embeddings in RAM — ~350 MB.
for _var in ("r1_model", "index", "corpus", "r1_rows"):
    if _var in globals():
        del globals()[_var]
gc.collect()
print("🧹 Stage 4 objects released")

🧹 Stage 4 objects released


---
## Stage 6 — Evaluate

Evaluates base, round-1, and round-2 models using **scene-level max-pool**:
each scene is embedded individually, and the movie score is the max cosine
similarity across its scenes. No Qdrant required — purely in-memory.

In [3]:
from pathlib import Path
import json
import logging

import finetuning.config as cfg
from finetuning.training.model import load_model
from finetuning.corpus.scene_corpus import build_scene_corpus
from finetuning.evaluation.scene_evaluator import run_scene_eval

log = logging.getLogger(__name__)

ROUND1_MODEL_DIR = "fede-embeddinggemma/round1"
ROUND2_MODEL_DIR = "fede-embeddinggemma/round2"
EVAL_MOVIES      = None
eval_path        = cfg.FINETUNING_DATA_DIR / "eval_queries.json"

eval_corpus = build_scene_corpus(max_movies=EVAL_MOVIES)

models_to_eval = {
    "base"   : None,              # None → loads config.EMBEDDING_MODEL_ID
    "round1" : ROUND1_MODEL_DIR,
    "round2" : ROUND2_MODEL_DIR,
}

results = {}
for name, model_path in models_to_eval.items():
    if model_path is not None and not Path(model_path).exists():
        log.warning("Skipping %s — directory not found", name)
        continue
    log.info("Evaluating %s …", name)
    m = load_model(model_path)
    metrics = run_scene_eval(m, eval_corpus, eval_path=eval_path)
    results[name] = metrics["summary"]
    del m

# Pretty-print comparison table
print(f"\n{'Model':<10} {'MRR':>6}  {'Acc@5':>6}  {'Acc@10':>7}  {'Acc@20':>7}")
print("-" * 45)
for name, s in results.items():
    print(
        f"{name:<10} {s.get('mrr', 0):>6.3f}  "
        f"{s.get('accuracy_at_5', 0):>6.3f}  "
        f"{s.get('accuracy_at_10', 0):>7.3f}  "
        f"{s.get('accuracy_at_20', 0):>7.3f}"
    )

# Save report
report_path = cfg.FINETUNING_DATA_DIR / "eval_report.json"
report_path.write_text(json.dumps(results, indent=2, ensure_ascii=False))
print(f"\n✅ Stage 6 complete — report saved to {report_path}")

c:\Users\anzez\University\ML\fede\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2026-04-09 01:28:41,726 [INFO] Scene corpus built: 1262 movies, 123397 total scenes
2026-04-09 01:28:41,729 [INFO] Evaluating base …
2026-04-09 01:28:41,729 [INFO] Loading embedding model: google/embeddinggemma-300m
2026-04-09 01:28:41,732 [INFO] Use pytorch device_name: cuda:0
2026-04-09 01:28:41,732 [INFO] Load pretrained SentenceTransformer: google/embeddinggemma-300m
2026-04-09 01:28:42,160 [INFO] HTTP Request: HEAD https://huggingface.co/google/embeddinggemma-300m/resolve/main/modules.json "HTTP/1.1 200 OK"
2026-04-09 01:28:42,292 [INFO] HTTP Request: HEAD https://huggingface.co/google/embeddinggemma-300m/resolve/main/config_sentence_transformers.json "HTTP/1.1 200 OK"
2026-04-09 01:28:42,421 [INFO] HTTP Request: HEAD http

  Encoding 123397 scenes from 1262 movies …


Batches: 100%|██████████| 1929/1929 [09:11<00:00,  3.50it/s]
2026-04-09 01:38:04,535 [INFO] Running scene-pool evaluation: 184 valid queries, top_k=20


  Embedding shape: (123397, 768), dtype: float32
  First 5 norms: [1.0018362 1.0037105 1.0015717 1.0035138 1.0010163]
  Eval queries: 184 total, 184 valid (target in corpus), 0 skipped
  Sample query: A young woman tells her boyfriend she's pregnant, and while he's initially overj…
  Target: trap_livin' | found in top-20: False
    #1 back_up_plan,_the                   score=0.4862
    #2 gone_girl                           score=0.4568
    #3 knocked_up                          score=0.4542


2026-04-09 01:38:08,096 [INFO]   evaluated 25 / 184 queries
2026-04-09 01:38:11,988 [INFO]   evaluated 50 / 184 queries
2026-04-09 01:38:15,795 [INFO]   evaluated 75 / 184 queries
2026-04-09 01:38:19,454 [INFO]   evaluated 100 / 184 queries
2026-04-09 01:38:23,017 [INFO]   evaluated 125 / 184 queries
2026-04-09 01:38:26,584 [INFO]   evaluated 150 / 184 queries
2026-04-09 01:38:30,443 [INFO]   evaluated 175 / 184 queries
2026-04-09 01:38:32,164 [INFO] Scene-pool evaluation complete — MRR=0.4349, Acc@5=0.543
2026-04-09 01:38:32,185 [INFO] Evaluating round1 …
2026-04-09 01:38:32,185 [INFO] Loading embedding model: fede-embeddinggemma/round1
Loading weights: 100%|██████████| 314/314 [00:00<00:00, 4012.15it/s]
2026-04-09 01:38:36,696 [INFO] Use pytorch device_name: cuda:0
2026-04-09 01:38:36,840 [INFO] Model loaded — device=cuda:0, dim=768


...

  Encoding 123397 scenes from 1262 movies …


Batches: 100%|██████████| 1929/1929 [09:26<00:00,  3.40it/s]
2026-04-09 01:58:13,245 [INFO] Running scene-pool evaluation: 184 valid queries, top_k=20


  Embedding shape: (123397, 768), dtype: float32
  First 5 norms: [1.0015168  0.9991372  0.99933654 0.99987096 1.0000168 ]
  Eval queries: 184 total, 184 valid (target in corpus), 0 skipped
  Sample query: A young woman tells her boyfriend she's pregnant, and while he's initially overj…
  Target: trap_livin' | found in top-20: True
    #1 back_up_plan,_the                   score=0.5895
    #2 knocked_up                          score=0.5728
    #3 margot_at_the_wedding               score=0.5612


2026-04-09 01:58:17,116 [INFO]   evaluated 25 / 184 queries
2026-04-09 01:58:21,138 [INFO]   evaluated 50 / 184 queries
2026-04-09 01:58:25,087 [INFO]   evaluated 75 / 184 queries
2026-04-09 01:58:28,852 [INFO]   evaluated 100 / 184 queries
2026-04-09 01:58:32,486 [INFO]   evaluated 125 / 184 queries
2026-04-09 01:58:36,083 [INFO]   evaluated 150 / 184 queries
2026-04-09 01:58:39,779 [INFO]   evaluated 175 / 184 queries
2026-04-09 01:58:41,258 [INFO] Scene-pool evaluation complete — MRR=0.5860, Acc@5=0.690



Model         MRR   Acc@5   Acc@10   Acc@20
---------------------------------------------
base        0.435   0.543    0.620    0.707
round1      0.564   0.663    0.761    0.810
round2      0.586   0.690    0.766    0.821

✅ Stage 6 complete — report saved to C:\Users\anzez\University\ML\fede\data\finetuning\eval_report.json


---
## Stage 6b — Upload model to Hugging Face

Pushes the final round-2 LoRA adapter (and Dense projection layers, tokenizer, configs) to a Hugging Face Hub repository

In [13]:
import os
from huggingface_hub import login
from finetuning.training.model import load_model

HF_REPO = "AnzeZ/fede-embeddinggemma"  # ← change this
PRIVATE = True

login(token=os.getenv("HF_TOKEN"))

model = load_model(ROUND2_MODEL_DIR)
model.push_to_hub(HF_REPO, private=PRIVATE)

print(f"\n✅ Stage 6b complete: model pushed to https://huggingface.co/{HF_REPO}")

2026-04-08 19:58:17,145 [INFO] HTTP Request: GET https://huggingface.co/api/whoami-v2 "HTTP/1.1 200 OK"
Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.
2026-04-08 19:58:17,150 [WARNING] Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.
2026-04-08 19:58:17,151 [INFO] Loading embedding model: fede-embeddinggemma/round2
Loading weights: 100%|██████████| 314/314 [00:00<00:00, 3490.52it/s]
2026-04-08 19:58:20,758 [INFO] Use pytorch device_name: cuda:0
2026-04-08 19:58:20,840 [INFO] Model loaded — device=cuda:0, dim=768
2026-04-08 19:58:21,150 [INFO] HTTP Request: POST https://huggingface.co/api/repos/create "HTTP/1.1 200 OK"
2026-04-08 19:58:21,152 [INFO] Save model to C:\Users\anzez\AppData\Local\Temp\tmplb6t_8rq
2026-04-08 19:58:21,295 [INFO] HTTP Request: HEAD https://huggingface.co/google/embeddinggemma-300m/resolve/main/config.j


✅ Stage 6b complete: model pushed to https://huggingface.co/AnzeZ/fede-embeddinggemma


---
## Stage 7 — Index corpus into Qdrant

Embeds all scenes with the final model and upserts them into Qdrant.

⚠️ Make sure Qdrant is running before executing this cell.

In [ ]:
from finetuning.training.model import encode_documents
from vector_db.indexer import SceneRecord, ScriptIndexer

final_model = load_model(ROUND2_MODEL_DIR)
corpus      = build_scene_corpus()  # full corpus, no max_movies
indexer     = ScriptIndexer()

total_scenes = sum(len(e.scenes) for e in corpus.values())
log.info("Indexing %d movies, %d scenes …", len(corpus), total_scenes)

indexed = 0
for i, (movie_id, entry) in enumerate(corpus.items(), 1):
    scene_texts = [s.text for s in entry.scenes]
    embeddings  = encode_documents(final_model, scene_texts, batch_size=64)

    records = [
        SceneRecord(
            movie_id=scene.movie_id,
            movie_title=scene.movie_title,
            scene_id=scene.scene_id,
            scene_index=scene.scene_index,
            text=scene.text,
            embedding=emb.tolist(),
            scene_title=scene.scene_title,
            character_names=scene.character_names,
        )
        for scene, emb in zip(entry.scenes, embeddings)
    ]

    indexer.index_movie_batch(scenes=records, sentences=[])
    indexed += len(records)

    if i % 100 == 0:
        log.info("  %d / %d movies indexed (%d scenes)", i, len(corpus), indexed)

print(f"\n✅ Stage 7 complete: {len(corpus)} movies, {indexed} scenes indexed")